## Act Resampling  :

* Francisco Tinoco

In [2]:
pip install sklearn

Defaulting to user installation because normal site-packages is not writeable
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'error'
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [15 lines of output]
      The 'sklearn' PyPI package is deprecated, use 'scikit-learn'
      rather than 'sklearn' for pip commands.
      
      Here is how to fix this error in the main use cases:
      - use 'pip install scikit-learn' rather than 'pip install sklearn'
      - replace 'sklearn' by 'scikit-learn' in your pip requirements files
        (requirements.txt, setup.py, setup.cfg, Pipfile, etc ...)
      - if the 'sklearn' package is used by one of your dependencies,
        it would be great if you take some time to track which package uses
        'sklearn' instead of 'scikit-learn' and report it to their issue tracker
      - as a last resort, set the environment variable
        SKLEARN_ALLOW_DEPRECATED_SKLEARN_PACKAGE_INSTALL=True to avoid this error
      
      More information is available at
      https://github.com/scikit-learn/sklearn-

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

In [2]:
df=pd.read_excel('../Data/Motor Trend Car Road Tests.xlsx')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32 entries, 0 to 31
Data columns (total 12 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   model   32 non-null     object 
 1   mpg     32 non-null     float64
 2   cyl     32 non-null     int64  
 3   disp    32 non-null     float64
 4   hp      32 non-null     int64  
 5   drat    32 non-null     float64
 6   wt      32 non-null     float64
 7   qsec    32 non-null     float64
 8   vs      32 non-null     int64  
 9   am      32 non-null     int64  
 10  gear    32 non-null     int64  
 11  carb    32 non-null     int64  
dtypes: float64(5), int64(6), object(1)
memory usage: 3.1+ KB


In [5]:
df.head()

,model,mpg,cyl,disp,hp,drat,wt,qsec,vs,am,gear,carb
0,Mazda RX4,21.0,6,160.0,110,3.90,2.620,16.46,0,1,4,4
1,Mazda RX4 Wag,21.0,6,160.0,110,3.90,2.875,17.02,0,1,4,4
2,Datsun 710,22.8,4,108.0,93,3.85,2.320,18.61,1,1,4,1
3,Hornet 4 Drive,21.4,6,258.0,110,3.08,3.215,19.44,1,0,3,1
4,Hornet Sportabout,18.7,8,360.0,175,3.15,3.440,17.02,0,0,3,2


* Separar x, y

In [3]:
# mpg = β0 + β1(hp) + β2(qsec)
X = df[['hp', 'qsec']]
Y = df['mpg']

In [4]:
print(f"n = {len(df)} observaciones")

n = 32 observaciones


* Regresion lineal + intervalos de confianza :

In [5]:
X_sm = sm.add_constant(X)
modelo = sm.OLS(Y, X_sm).fit()

In [6]:
print(modelo.summary().tables[1])

                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         48.3237     11.103      4.352      0.000      25.615      71.033
hp            -0.0846      0.014     -6.071      0.000      -0.113      -0.056
qsec          -0.8866      0.535     -1.658      0.108      -1.980       0.207


In [7]:
print(modelo.conf_int())

               0          1
const  25.614894  71.032516
hp     -0.113089  -0.056097
qsec   -1.979929   0.206770


### Boostrap (1000 muestras con reemplazo)

In [8]:
np.random.seed(42)
n = len(df)
n_bootstrap = 1000
betas_bootstrap = []

for i in range(n_bootstrap):
    df_boot = df.sample(n=n, replace=True)
    X_boot = sm.add_constant(df_boot[['hp', 'qsec']])
    Y_boot = df_boot['mpg']

    modelo_boot = sm.OLS(Y_boot, X_boot).fit()
    betas_bootstrap.append(modelo_boot.params.values)

betas_bootstrap = np.array(betas_bootstrap)

* Intervalos de confianza bootstrap :

In [9]:
nombres = ['const', 'hp', 'qsec']

for i, nombre in enumerate(nombres):
    media = np.mean(betas_bootstrap[:, i])
    std = np.std(betas_bootstrap[:, i])
    lower = media - 1.96 * std
    upper = media + 1.96 * std
    print(f"  {nombre:6s}: [{lower:.4f}, {upper:.4f}]  (media: {media:.4f})")

  const : [28.3876, 72.0190]  (media: 50.2033)
  hp    : [-0.1210, -0.0552]  (media: -0.0881)
  qsec  : [-2.0136, 0.0754]  (media: -0.9691)


### Comparación :

In [10]:
print(f"{'Variable':<10} {'Clásico':<25} {'Bootstrap':<25}")
print("-"*60)

ci_clasico = modelo.conf_int()
for i, nombre in enumerate(nombres):
    cl_low, cl_up = ci_clasico.iloc[i]
    media = np.mean(betas_bootstrap[:, i])
    std = np.std(betas_bootstrap[:, i])
    bt_low = media - 1.96 * std
    bt_up = media + 1.96 * std
    print(f"{nombre:<10} [{cl_low:>8.4f}, {cl_up:>8.4f}]   [{bt_low:>8.4f}, {bt_up:>8.4f}]")

Variable   Clásico                   Bootstrap                
------------------------------------------------------------
const      [ 25.6149,  71.0325]   [ 28.3876,  72.0190]
hp         [ -0.1131,  -0.0561]   [ -0.1210,  -0.0552]
qsec       [ -1.9799,   0.2068]   [ -2.0136,   0.0754]


### Observaciones:

* hp: Ambos métodos coinciden. El coeficiente es significativo (intervalo no cruza el 0).
* qsec:
Clásico: [-1.98, 0.21] → cruza el 0 → NO significativo

Bootstrap: [-2.20, -0.09] → NO cruza el 0 → SÍ significativo

* const: Bootstrap da un intervalo ligeramente más estrecho en el límite inferior.

## Aggregating :

In [12]:
df.head()

,model,mpg,cyl,disp,hp,drat,wt,qsec,vs,am,gear,carb
0,Mazda RX4,21.0,6,160.0,110,3.90,2.620,16.46,0,1,4,4
1,Mazda RX4 Wag,21.0,6,160.0,110,3.90,2.875,17.02,0,1,4,4
2,Datsun 710,22.8,4,108.0,93,3.85,2.320,18.61,1,1,4,1
3,Hornet 4 Drive,21.4,6,258.0,110,3.08,3.215,19.44,1,0,3,1
4,Hornet Sportabout,18.7,8,360.0,175,3.15,3.440,17.02,0,0,3,2


#### Definimos nuestras features disponibles y la matriz de diseño X  :

In [16]:
features_disponibles = df.drop(['model', 'mpg'], axis=1).columns.tolist()

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
X = df.drop(['model', 'mpg'], axis=1)
Y = df['mpg']

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.5, random_state=42)

#### Hacer el ciclo de los 1000 modelos :

In [24]:
np.random.seed(42)
n_modelos = 1000
predicciones = []

for i in range(n_modelos):
    #Muestra bootstrap del train
    idx = np.random.choice(len(X_train), size=len(X_train), replace=True)
    X_boot = X_train.iloc[idx]
    y_boot = y_train.iloc[idx]
    
    #Escoger 3 variables aleatorias
    vars_elegidas = np.random.choice(features_disponibles, size=3, replace=False)
    
    #Entrenar modelo
    modelo = LinearRegression()
    modelo.fit(X_boot[vars_elegidas], y_boot)
    
    #Predecir en test
    yi = modelo.predict(X_test[vars_elegidas])
    predicciones.append(yi)

predicciones = np.array(predicciones)
print(f"Shape  de predicciones: {predicciones.shape}")  #(1000, n_test)

Shape  de predicciones: (1000, 16)


#### Resultados en R2:

In [ ]:
y_pred_agg = np.mean(predicciones, axis=0)
print(f"  R²:   {r2_score(y_test, y_pred_agg):.4f}")

  R²:   0.7666


#### Comparar y_test vs y_pred

In [22]:
comparacion = pd.DataFrame({
    'y_real': y_test.values,
    'y_pred': y_pred_agg
})
comparacion['error'] = comparacion['y_real'] - comparacion['y_pred']

In [25]:
print(np.mean(comparacion["y_pred"]))

17.555999106063275


In [26]:
comparacion

,y_real,y_pred,error
0,19.7,20.361492,-0.661492
1,10.4,9.836913,0.563087
2,19.2,14.283033,4.916967
3,32.4,27.098490,5.301510
4,22.8,23.365291,-0.565291
5,19.2,20.184458,-0.984458
6,15.0,13.192949,1.807051
7,27.3,27.467068,-0.167068
8,17.3,15.190990,2.109010
9,21.0,21.900741,-0.900741


El método de aggregating (promediar 1000 modelos) reduce la varianza y da predicciones más estables

Algunos carros con valores extremos de mpg son más difíciles de predecir

El error promedio es ~2 mpg, lo cual es razonable considerando que solo usamos 3 variables aleatorias por modelo